# Keystroke Dynamics — 1D CNN Classification Web App

**Backend built with FastAPI and PyTorch.**

This notebook walks through the complete code for a keystroke dynamics authentication system. The idea is simple:
- Users **register** by typing a phrase — we capture *hold times* (how long each key is pressed) and *flight times* (gap between consecutive key presses).
- A **1D CNN** model is trained to classify users based on their unique typing patterns.
- During **prediction**, the model identifies who is typing (or says "Unknown User" if confidence is low).

### API Endpoints
| Endpoint | Method | Description |
|----------|--------|-------------|
| `/` | GET | Serves the frontend HTML page |
| `/register` | POST | Register a user with keystroke timing data and retrain the model |
| `/predict` | POST | Predict who is typing based on keystroke timing data |

## 1. Import Required Libraries

We need:
- **FastAPI** — for the web server and API endpoints
- **PyTorch** — for building and training the 1D CNN model
- **NumPy** — for array manipulation and preprocessing
- **Pydantic** — for request/response validation schemas

In [ ]:
import json
import os
import numpy as np
from fastapi import FastAPI
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse
from pydantic import BaseModel
from typing import List, Optional
from contextlib import asynccontextmanager

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim

## 2. Configuration & Constants

- **`DATA_FILE`** — JSON file where keystroke samples are persisted between server restarts.
- **`SEQUENCE_LENGTH`** — Each typing sample is padded/truncated to 20 keystroke pairs `[hold_time, flight_time]`.
- **`CONFIDENCE_THRESHOLD`** — If the model's confidence is below 65%, we label the prediction as "Unknown User".

In [ ]:
# Persistence file
DATA_FILE = "keystroke_data.json"

# In-memory dataset & model storage
dataset: List[dict] = []
model: Optional[nn.Module] = None
label_map: dict = {}        # {username: int}

# Hyperparameters
SEQUENCE_LENGTH = 20         # Fixed sequence length for the 1D CNN
CONFIDENCE_THRESHOLD = 0.65  # Below this → "Unknown User"

## 3. Helper Functions

### 3a. Dataset Persistence

Two simple functions to save/load the keystroke dataset to/from a JSON file so data survives server restarts.

In [ ]:
def save_dataset():
    """Persist the in-memory dataset to a JSON file."""
    with open(DATA_FILE, "w") as f:
        json.dump(dataset, f)

def load_dataset():
    """Load dataset from disk if the file exists."""
    if os.path.exists(DATA_FILE):
        with open(DATA_FILE, "r") as f:
            return json.load(f)
    return []

### 3b. Request / Response Schemas (Pydantic)

These Pydantic models validate the JSON request bodies:
- **`KeystrokeData`** — holds the two timing arrays (`hold_times`, `flight_times`).
- **`RegisterRequest`** — wraps a username + keystroke data for registration.
- **`PredictRequest`** — wraps keystroke data for prediction.

In [ ]:
class KeystrokeData(BaseModel):
    hold_times: List[float]
    flight_times: List[float]


class RegisterRequest(BaseModel):
    username: str
    keystrokes: KeystrokeData


class PredictRequest(BaseModel):
    keystrokes: KeystrokeData

### 3c. Preprocessing — Pad/Truncate Keystroke Features

Raw keystroke data comes in as variable-length lists. We need a **fixed-size input** for the CNN, so we:
1. Pair up `hold_time[i]` and `flight_time[i]` into `[hold, flight]` pairs.
2. **Pad** with `[0.0, 0.0]` if shorter than `SEQUENCE_LENGTH` (20).
3. **Truncate** if longer.

Output shape: `(1, 20, 2)` — one sample, 20 timesteps, 2 features per timestep.

In [ ]:
def preprocess(keystrokes: KeystrokeData) -> np.ndarray:
    """Pad/truncate keystroke features to (SEQUENCE_LENGTH, 2)."""
    hold = keystrokes.hold_times
    flight = keystrokes.flight_times

    # Build pairs [hold_time, flight_time] for each keystroke
    length = min(len(hold), len(flight))
    features = [[hold[i], flight[i]] for i in range(length)]

    # Pad or truncate to SEQUENCE_LENGTH
    if len(features) < SEQUENCE_LENGTH:
        features += [[0.0, 0.0]] * (SEQUENCE_LENGTH - len(features))
    else:
        features = features[:SEQUENCE_LENGTH]

    return np.array(features, dtype=np.float32).reshape(1, SEQUENCE_LENGTH, 2)

## 4. The 1D CNN Model (PyTorch)

A lightweight **1D Convolutional Neural Network** architecture:

```
Input (batch, 20, 2)
  → permute to (batch, 2, 20)   # Conv1d expects channels-first
  → Conv1d(2→16, kernel=3, pad=1)
  → ReLU
  → MaxPool1d(kernel=2)          # output: (batch, 16, 10)
  → Flatten                      # output: (batch, 160)
  → Linear(160 → num_classes)    # raw logits
```

The model is small enough to train in seconds on CPU, which is perfect for a real-time web app.

In [ ]:
class KeystrokeCNN(nn.Module):
    """Lightweight 1D CNN for keystroke-based user classification."""

    def __init__(self, num_classes: int):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=2, out_channels=16,
                               kernel_size=3, padding=1)  # same padding
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        # After conv+pool: (16, SEQUENCE_LENGTH//2) → flattened = 16 * 10 = 160
        self.flatten_size = 16 * (SEQUENCE_LENGTH // 2)
        self.fc = nn.Linear(self.flatten_size, num_classes)

    def forward(self, x):
        # x shape: (batch, seq_len, 2) → transpose to (batch, 2, seq_len) for Conv1d
        x = x.permute(0, 2, 1)
        x = self.pool(self.relu(self.conv1(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x  # raw logits; softmax applied during inference

## 5. Training — Build & Train the 1D CNN

This function is called every time a new user registers. It:
1. Builds a fresh **label map** from all unique usernames.
2. Preprocesses every stored sample into a padded `(N, 20, 2)` tensor.
3. Creates a new `KeystrokeCNN` model and trains it for **30 epochs** using:
   - **Loss:** `CrossEntropyLoss`
   - **Optimizer:** `Adam` (lr=0.001)
   - **Batch size:** 4
4. Shuffles data each epoch for better generalization.

In [ ]:
def build_and_train_model():
    """Build a 1D CNN and train on all registered keystroke samples."""
    global model, label_map

    # Build label map from unique usernames
    usernames = sorted(set(entry["username"] for entry in dataset))
    label_map = {name: idx for idx, name in enumerate(usernames)}
    num_classes = len(usernames)

    # Prepare training arrays
    X_list, y_list = [], []
    for entry in dataset:
        X_list.append(preprocess(
            KeystrokeData(
                hold_times=entry["hold_times"],
                flight_times=entry["flight_times"],
            )
        ))
        y_list.append(label_map[entry["username"]])

    X_train = np.concatenate(X_list, axis=0)   # (N, 20, 2)
    y_train = np.array(y_list, dtype=np.int64)  # (N,)

    # Convert to PyTorch tensors
    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.long)

    # 1D CNN Architecture (lightweight — trains in seconds)
    model = KeystrokeCNN(num_classes)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Train quickly (few epochs, small dataset)
    model.train()
    batch_size = 4
    n_samples = X_tensor.size(0)

    for epoch in range(30):
        # Shuffle each epoch
        perm = torch.randperm(n_samples)
        X_shuffled = X_tensor[perm]
        y_shuffled = y_tensor[perm]

        for i in range(0, n_samples, batch_size):
            X_batch = X_shuffled[i:i + batch_size]
            y_batch = y_shuffled[i:i + batch_size]

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

    model.eval()

## 6. Load Data & Set Up FastAPI App

On startup the app:
1. Loads any previously saved keystroke data from `keystroke_data.json`.
2. If data exists, retrains the model automatically.
3. Mounts the `static/` folder to serve the frontend (HTML + JS).

In [ ]:
@asynccontextmanager
async def lifespan(app: FastAPI):
    """Load persisted dataset and retrain model on startup."""
    global dataset
    dataset = load_dataset()
    if dataset:
        print(f"Loaded {len(dataset)} sample(s) from {DATA_FILE}")
        build_and_train_model()
        print("Model retrained from saved data.")
    else:
        print("No saved data found. Starting fresh.")
    yield

app = FastAPI(lifespan=lifespan)

# Serve static files (index.html, app.js)
app.mount("/static", StaticFiles(directory="static"), name="static")

## 7. API Route Handlers

### `GET /` — Serve the Frontend
Returns the `index.html` page.

### `POST /register` — Register a User
- Validates the username (1–50 chars).
- Appends the keystroke sample to the dataset.
- Saves to disk and **retrains** the model.

### `POST /predict` — Identify Who is Typing
- Preprocesses the incoming keystroke data.
- Runs it through the trained CNN.
- Applies **softmax** to get class probabilities.
- Returns the predicted username and confidence %. If confidence < 65%, returns "Unknown User".

In [ ]:
@app.get("/")
async def serve_index():
    """Serve the main HTML page."""
    return FileResponse("static/index.html")


@app.post("/register")
async def register(req: RegisterRequest):
    """
    Register a new typing sample.
    Appends data to the global dataset and retrains the 1D CNN.
    """
    # Basic username validation
    if not req.username or len(req.username) > 50:
        return {"status": "error", "message": "Username must be 1–50 characters."}

    # Store the raw timing lists together with the username
    dataset.append({
        "username": req.username,
        "hold_times": req.keystrokes.hold_times,
        "flight_times": req.keystrokes.flight_times,
    })

    # Save dataset to disk & retrain the model
    save_dataset()
    build_and_train_model()

    return {
        "status": "ok",
        "message": f"User '{req.username}' registered. Model trained on {len(dataset)} sample(s).",
    }


@app.post("/predict")
async def predict(req: PredictRequest):
    """
    Predict the user from a typing sample using the trained 1D CNN.
    Returns 'Unknown User' when the model's confidence is below the threshold.
    """
    if model is None or len(label_map) == 0:
        return {"match": "Unknown User", "confidence": 0.0,
                "message": "No model trained yet. Please register users first."}

    X_test = preprocess(req.keystrokes)
    X_tensor = torch.tensor(X_test, dtype=torch.float32)

    with torch.no_grad():
        logits = model(X_tensor)
        probabilities = torch.softmax(logits, dim=1)[0]  # shape (num_classes,)

    max_prob = float(probabilities.max())
    predicted_idx = int(probabilities.argmax())

    # Reverse lookup: index → username
    idx_to_name = {v: k for k, v in label_map.items()}
    predicted_name = idx_to_name.get(predicted_idx, "Unknown User")

    if max_prob < CONFIDENCE_THRESHOLD:
        return {"match": "Unknown User", "confidence": round(max_prob * 100, 2)}

    return {"match": predicted_name, "confidence": round(max_prob * 100, 2)}

## 8. Run the Server

Start the FastAPI app with **Uvicorn** on `http://0.0.0.0:8000`.

Once running, open your browser to `http://localhost:8000` to use the keystroke dynamics app.

> **Note:** Running this cell will block the notebook. Stop the kernel to shut down the server.

In [ ]:
import uvicorn
uvicorn.run(app, host="0.0.0.0", port=8000)